In [16]:
import json
import os
from pathlib import Path

import requests

In [ ]:
url = "https://api.coresignal.com/cdapi/v2/employee_multi_source/search/es_dsl"

payload = json.dumps(
    {
        "query": {
            "bool": {
                "filter": [
                    {"term": {"is_deleted": 0}},
                    {"term": {"is_parent": 1}},
                ],
                "must": [
                    {"match_phrase": {"location_city": "Berlin"}},
                    {
                        "nested": {
                            "path": "experience",
                            "query": {
                                "bool": {
                                    "filter": [
                                        {"term": {"experience.active_experience": 1}},
                                        {
                                            "terms": {
                                                "experience.company_id": [
                                                    20706027,
                                                    9531567,
                                                ]
                                            }
                                        },
                                    ],
                                    "must": [
                                        {
                                            "bool": {
                                                "should": [
                                                    {
                                                        "match_phrase": {
                                                            "experience.position_title": "Data Scientist"
                                                        }
                                                    },
                                                    {
                                                        "match_phrase": {
                                                            "experience.position_title": "Data  Scientist"
                                                        }
                                                    },
                                                ],
                                                "minimum_should_match": 1,
                                            }
                                        }
                                    ],
                                }
                            },
                        }
                    },
                ],
            }
        },
        "sort": ["_score"],
    }
)

headers = {
    "Content-Type": "application/json",
    "apikey": os.environ["CORESIGNAL_API_KEY"],
}

response = requests.request("POST", url, headers=headers, data=payload)
persion_ids: list[int] = json.loads(response.text)

print(persion_ids)

In [ ]:
url = f"https://api.coresignal.com/cdapi/v2/employee_multi_source/collect/{persion_ids[0]}"

headers = {
    "Content-Type": "application/json",
    "apikey": os.environ["CORESIGNAL_API_KEY"],
}

response = requests.request("GET", url, headers=headers)
print(json.dumps(response.json(), indent=4))

In [18]:
# Write the response to a JSON file
output_file = (
    Path("..")
    / "spi"
    / "coresignal"
    / "employee_multi_source"
    / f"{persion_ids[0]}.json"
)

output_file.parent.mkdir(parents=True, exist_ok=True)
with open(output_file, "w") as f:
    json.dump(response.json(), f, indent=4)